<a href="https://colab.research.google.com/github/DM871/dm871.github.io/blob/master/notebooks/lab_highspy_1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## The Basics: Production Allocation

The following model is for a specific instance of the production 
allocation problem seen in the first lectures. We give here the primal
and its dual model with the instantiated numerical parameters.

$$\begin{array}{*{6}{r}l}
\max & 5x_1&+&6x_2&+&8x_3&=z\\
&6x_1&+&5x_2&+&10x_3 &\leq 60\\
&8x_1&+&4x_2&+&4x_3&\leq 40\\
&4x_1&+&5x_2&+&6x_3&\leq 50\\
&x_1,&&x_2,&&x_3&\geq 0
\end{array}$$

$$\begin{array}{*{6}{r}l}
\min & 60y_1&+&40y_2&+&50y_3&=u\\
&6y_1&+&8y_2&+&4y_3 &\leq 5\\
&5y_1&+&4y_2&+&5y_3&\leq 6\\
&10y_1&+&4y_2&+&6y_3&\leq 8\\
&y_1,&&y_2,&&y_3&\geq 0
\end{array}$$

## Analysis of the final tableau 

Solving one of the two problems provides the solution also to the other problem. The primal solution is
$x^*_1=0,x^*_2=7,x^*_3=2.5$ and the dual solution is
$y^*_1=0.2,y^*_2=0,y^*_3=1$. The objective value is $z^*=u^*=62$.

We can organize these values in a *tableau*. For the given problems it looks like this:

    |------+----+----+------+----+----+----+-----|
    |   x1 | x2 | x3 |   s1 | s2 | s3 | -z |   b |
    |------+----+----+------+----+----+----+-----|
    |   ?  |  1 |  0 |  ?   |  0 |  ? |  0 |   7 |
    |   ?  |  0 |  1 |  ?   |  0 |  ? |  0 | 5/2 |
    |   ?  |  0 |  0 |  ?   |  1 |  ? |  0 |   2 |
    |------+----+----+------+----+----+----+-----|
    | -0.2 |  0 |  0 | -0.2 |  0 | -1 |  1 | -62 |
    |------+----+----+------+----+----+----+-----|

A question mark is for the values that are not relevant for the goal of
this exercise.

The three numbers of the last row in the tableau above in the columns of
the variables that are not in basis are called *reduced costs*. They
indicate how much we should make each product more expensive in order to
be worth manufacturing it. The next three values are known as *shadow
prices*. After a change of sign they give the values of the dual
variables, which are interpreted as the *marginal values* of increasing
(or decreasing) the capacities of the resources (that is, the values by
which the objective function would improve if the corresponding constraints were
relaxed by one unit, which corresponds to buying one unit more of
that resource). In our example, which seeks maximization, the marginal value
1 for the third resource means that the objective function would
increase by 1 if we had one more unit of that resource.

It can be verified that in the primal problem at the optimum the first
and third resources are fully exhausted, that is, their constraint is
satisfied at the equality, while there is *slack* for the second
resource, that is, the constraint holds with strict inequality. Looking
at the marginal values, we see that the second resource has been given a
zero valuation. This seems plausible, since we are not using all the
capacity that we have, we are not willing to place much value on it
(buying one more unit of that resource would not translate in an
improvement of the objective function).

These results are captured by the Complementary Slackness theorem of
linear programming. If a constraint is not *"binding"* in the optimal
primal solution, the corresponding dual variable is zero in the optimal
solution to the dual model. Similarly, if a constraint in the dual model
is not *"binding"* in the optimal solution to the dual model, then the
corresponding variable is zero in the optimal solution to the primal
model.


## Solving the model with highspy

Let's write the primal model in Python and solve it with
[HiGHS](https://highs.dev), an open-source (MIT licence) solver for linear
programming (LP), mixed integer programming (MIP) and quadratic programming
problems. Its Python interface is called `highspy`. Since HiGHS is open
source, there are no license limitations on the size of the models. The
following command installs it (also in Google Colab):

In [ ]:
%pip install highspy

Here is the script:

In [ ]:
import highspy

inf = highspy.kHighsInf

# Model
model = highspy.Highs()

# Create decision variables
x1 = model.addVariable(lb=0.0, ub=inf, obj=5.0, name="x1")  # arguments by name
x2 = model.addVariable(0.0, inf, 6.0, name="x2")  # arguments by position
x3 = model.addVariable(name="x3")  # arguments by default

# The objective is to maximize (this will overwrite the obj given in the variable declaration)
model.setObjective(5.0*x1 + 6.0*x2 + 8.0*x3, sense=highspy.ObjSense.kMaximize)

# Add the constraints to the model
c1 = model.addConstr(6.0*x1 + 5.0*x2 + 10.0*x3 <= 60.0, name="c1")
c2 = model.addConstr(8.0*x1 + 4.0*x2 + 4.0*x3 <= 40.0, name="c2")
c3 = model.addConstr(4.0*x1 + 5.0*x2 + 6.0*x3 <= 50.0, name="c3")



The documentation for the methods `Highs.addVariable()` and
`Highs.addConstr()`, as well as for all other functions in `highspy`, is
available from the [HiGHS documentation](https://ergo-code.github.io/HiGHS/dev/)
and more specifically from the [Python interface
page](https://ergo-code.github.io/HiGHS/dev/interfaces/python/).
For the variable $x_3$ the lower bound, upper bound, objective
coefficient and type are set to their default values that are,
respectively: `lb=0.0, ub=highspy.kHighsInf, obj=0.0` and continuous.
(Integer variables can be declared with `type=highspy.HighsVarType.kInteger`
or with `Highs.addIntegral()` and `Highs.addBinary()`.)
Once the model has been built, it is good practice to inspect it to make
sure that it has been built as we intended. A convenient way is to write
the model to a file in LP format and print it:

In [ ]:
model.writeModel("prod.lp")
print(open("prod.lp").read())

We are then ready to solve or optimize the model
(using `model.run()`). You can then query the value of the
variables with `model.val()` (and their names with `model.variableName()`).

In [ ]:
# Solve
model.run()

# Let's print the solution
print(model.modelStatusToString(model.getModelStatus()))
print("Objective value:", model.getObjectiveValue())
for v in model.getVariables():
    print(model.variableName(v), model.val(v))

The screen output of HiGHS is described in the
[documentation](https://ergo-code.github.io/HiGHS/dev/). HiGHS
reports first some statistics on the ranges of the coefficients of the
problem, which are useful to check whether there is a need for
rescaling conveniently the numbers of the problem (no need in this case),
and then applies preprocessing techniques (presolve) to assess whether the model
can be trivially solved without running the simplex or whether it can be
reduced. In our case none of the two are possible and we are left with 3
rows and 3 columns. Finally, the simplex method is applied (by default the dual simplex)
and after a few iterations the optimal solution is found with value 62.
In the log, the column `Infeasibilities num(sum)` reports the number and the sum of
primal (`Pr`) and dual (`Du`) infeasibilities; `Ph1` indicates that the
algorithm is in phase I.

In `highspy`, the model, the solution and the information about the solution
process are retrieved via methods of the `Highs` object:

- `model.getModelStatus()` returns the status of the model after solving
  (eg, `highspy.HighsModelStatus.kOptimal`, `kInfeasible`, `kUnbounded`,
  `kUnboundedOrInfeasible`);
- `model.getSolution()` returns an object with the lists `col_value`
  (values of the variables), `col_dual` (reduced costs), `row_value`
  (values of the left-hand sides of the constraints) and `row_dual` (dual values);
- `model.getInfo()` returns information about the solution process, eg,
  `objective_function_value` and `simplex_iteration_count`;
- `model.getLp()` returns the model data (`col_cost_`, `col_lower_`, `row_upper_`, ...);
- for single variables and constraints there are the convenience methods
  `model.val(x)`, `model.variableDual(x)`, `model.constrValue(c)` and
  `model.constrDual(c)`.

The data of a model can be modified after its construction, for example with
`model.changeColCost()`, `model.changeColBounds()`, `model.changeRowBounds()`
and `model.changeCoeff()`. Note that HiGHS stores each constraint in the form
$L \leq a^Tx \leq U$, hence the right-hand side of a $\leq$ constraint is its upper bound $U$.
For example, `model.changeRowBounds(c1.index, -inf, 61)` changes the right-hand side
of the first constraint to 61.

Note that the code above can also be put in a script file called `prod1.py`. Then we can solve the model
by calling:

```
    > python prod1.py
```
If something goes wrong check that `highspy` is available at the
import and that there are no syntax errors.

#### The Value of the Dual variables

The value of the dual variables can be accessed by the method `constrDual` applied to the
corresponding constraints or, all at once, via the `row_dual` list of the solution. In Python:

In [ ]:
# Let's print the dual variables
for c in model.getConstrs():
    print(model.getRowName(c.index)[1], model.constrDual(c))

print(model.getSolution().row_dual)

The values `c1`,`c2`,`c3` are the shadow prices (here 0.2, 0.0 and 1) which correspond to
the marginal value of the resources. The c1 and c3 constraints' value is
different from zero. This indicates that there's a variable on the upper
bound for those constraints, or in other terms that these constraints
are *"binding"*. The second constraint is not *"binding"*.

#### Your Task

Try relaxing the value of each binding constraint one at a time, solve
the modified problem, and see what happens to the optimal value of the
objective function. Also check that, as expected, changing the value of
non-binding constraints won't make any difference to the solution.

#### Your Task

We can also access several quantities associated with the variables. A
particularly relevant one is the *reduced cost*. Print the reduced costs
of the variables for our example and make sure that they correspond to
the expected values from the tableau above. \[Hint: look for the
method `variableDual` or the list `col_dual` of the solution.\] What can we say about the solution found on
the basis of the reduced costs?

Let's now focus on the values output during the execution of the
simplex. Let's first solve another small numerical example:


In [ ]:
import highspy

m = highspy.Highs()

x = m.addVariable(name="x") # ie, >= 0
y = m.addVariable(name="y") # ie, >= 0

m.setObjective(x - y, sense=highspy.ObjSense.kMaximize)

m.addConstr(x + y <= 2, name="c1")
m.addConstr(2*x + 2*y >= 5, name="c2")

m.run()
print(m.modelStatusToString(m.getModelStatus()))


This means that the presolve process has identified the model as
infeasible, without running the simplex. In other cases, presolve may
only be able to conclude that the model is infeasible *or* unbounded
(status `kUnboundedOrInfeasible`). To see what the simplex does we switch
off the presolve process and resolve. We also ensure that we are using the
primal simplex that makes things easier to interpret for us.

HiGHS allows to control these choices by means of options. They
include time limits or other execution halting controls. See
[Options](https://ergo-code.github.io/HiGHS/dev/options/definitions/)
for a reference list. Options are set with `m.setOptionValue(name, value)`.
The option `presolve` controls whether presolving is used (`"off"`, `"choose"`, `"on"`).
The option `solver` controls which solution method is used (`"simplex"`,
`"ipm"` for the interior point method, `"pdlp"`, or `"choose"`, the default).
The option `simplex_strategy` controls which simplex is used. The default is
the dual simplex. The possibilities are 0=automatic, 1=dual simplex (serial),
2=dual simplex (PAMI), 3=dual simplex (SIP), 4=primal simplex.
Let's add the following lines to our second small model.
The method `m.clearSolver()` discards the previous solution and basis so
that the model is solved from scratch.

In [ ]:
if m.getModelStatus() in (highspy.HighsModelStatus.kInfeasible,
                          highspy.HighsModelStatus.kUnboundedOrInfeasible):
    # Turn presolve off to see the simplex at work
    m.setOptionValue("presolve", "off")
    m.setOptionValue("solver", "simplex")
    m.setOptionValue("simplex_strategy", 4) # 0=automatic, 1=dual (serial), 2=dual (PAMI), 3=dual (SIP), 4=primal
    m.clearSolver()
    m.run()

status = m.getModelStatus()
if status == highspy.HighsModelStatus.kOptimal:
    print('Optimal objective: %g' % m.getObjectiveValue())
    print(m.val(x))
    print(m.val(y))
elif status != highspy.HighsModelStatus.kInfeasible:
    print('Optimization was stopped with status %s' % m.modelStatusToString(status))
else:
    print('The model is infeasible')

Resolving gives us the needed information. The value reported under `Ph1:`
in the column `Infeasibilities num(sum)` is the number and the sum of the
infeasibilities, that is, the objective function of the auxiliary problem
in the phase I of the two-phase simplex method. It never reaches 0. This means
that a feasible solution cannot be found and the problem is therefore
infeasible. Try to change 5 to 2 in the right-hand-side of the second
constraint of the model above. What happens? Explain the behaviour.



#### Your Task

Resolve the production allocation example above using the primal method.
Compare the iteration values of the simplex with the previous ones. Give
your interpretation of the values Objective and Infeasibilities, primal (Pr) and dual (Du).



In [ ]:
model.clearSolver()
model.setOptionValue("presolve", "off")
model.setOptionValue("solver", "simplex")
model.setOptionValue("simplex_strategy", 4) # 0=automatic, 1=dual (serial), 2=dual (PAMI), 3=dual (SIP), 4=primal
model.run();

#### Exporting Model Data to a File

The method `model.writeModel(filename)` writes model data to a file. The file type
is encoded in the suffix of the file name passed to the function. Valid
suffixes for writing the model itself are `.mps` and `.lp`.
The method `model.writeSolution(filename, style)` writes the solution
(use `style=1` for a human readable format) while `model.writeOptions(filename)`
writes the options to a file and `model.writeBasis(filename)` the final basis.
Conversely, `model.readModel(filename)` reads a model from an `.mps` or `.lp` file.

#### Your Task

Try all these formats on the production allocation example above and
check their contents. The MPS file is not very user friendly. This is
because the format is an old format when computer technology had much
more limitations than nowadays. The CPLEX-LP format is a more explicit
version of the problem that may be useful to check whether the model we
implemented in Python is actually the one we intended.

If you have any of them installed, try solving the problem with other
solvers, eg, `cplex`, `glpk` and `scip`. For this you have to use the
MPS (Mathematical Programming System) format and run the following:
```
    cplex -c read prod.mps optimize
    glpsol --mps prod.mps
    scip -f prod.mps
```
HiGHS has also a [command-line
executable](https://ergo-code.github.io/HiGHS/dev/executable/)
to solve model files:
```
    highs model.mps
```
You may also use the online solver at NEOS, the Network Enabled
Optimization Server supported by the US federal government and located
at Argonne National Lab. To submit an MPS model to NEOS visit
<http://www.neos-server.org/neos/>, click on the icon "NEOS Solvers",
scroll down to the Linear Programming or Mixed Integer Linear
Programming list, click on one of those, scroll down to "Model File",
click on "Choose File", select a file from your computer that contains
an MPS model, scroll down to "e-mail address:", type in your email
address, and click Submit to NEOS.

    [Standard MPS files do not indicate whether to minimize or maximize
    the objective. Thus your MPS files will come out the same whether
    the objective is minimize or maximize.

    As you are seeing, most solvers minimize the objective by default. A
    solver may have a switch to tell it to maximize instead, but that is
    different for each solver.

    If you change the signs of all the objective coefficients, while
    leaving the constraints unchanged, then minimizing the resulting MPS
    file will be equivalent to maximizing the original problem. This can
    be done easily by putting the entire objective expression in
    parentheses and placing a minus sign in front of it.]